# Training FNO, MRBNO, and PCA Net on Kolmogorov Flow Dataset

Train and compare three models:
1. **FNO** (Fourier Neural Operator) - layers=4, width=24, modes=12
2. **MRBNO Lift v3** (Multi-layer Reduced Basis Neural Operator with lift architecture) - layers=4, width=24
3. **PCA Net** (Simple neural network operating in PCA coefficient space) - depth=5, hidden=1024

on the 2D Kolmogorov Flow (Navier-Stokes) with Reynolds number Re=5000.

**Dataset Structure:**
- Format: u(t) -> u(t+1) (single-timestep prediction)
- Input: u(t) - velocity field at time t (1 channel, 128x128)
- Output: u(t+1) - velocity field at time t+1 (1 channel, 128x128)
- Dataset: `/pscratch/sd/d/dvod/2D_NS_Re5000.npy` (100 trajectories, 501 timesteps each)
- Total pairs: 50,000 (requires ~3 GB RAM)

**⚠️ IMPORTANT: Resource Requirements:**
- **RAM:** ~3 GB for data (train/val), ~8-16 GB GPU memory for training
- **Recommendation:** For login nodes, consider submitting via SLURM to compute nodes
- This notebook uses memory-efficient chunked loading to minimize RAM usage

**Note:** PCA Net and MRBNO Lift v3 use the same PCA reducer to ensure fair comparison.

**Training settings:**
- Cosine annealing LR schedule with lr=2e-3
- Gradient clipping: max_grad_norm=3.0
- Early stopping: patience=50
- Batch size: 8 for all models (128x128 resolution requires reduced batch size)

**⚠️ IMPORTANT: MRBNO Training:**
- MRBNO is too large to train in this notebook (3.5B parameters, causes OOM)
- Use the separate training script: `train_mrbno_kolmogorov_Re5000.py`
- Example: `python train_mrbno_kolmogorov_Re5000.py --batch_size 8`
- The script will save checkpoint to `checkpoints/mrbno_lift_v3_kolmogorov_Re5000_best.pt`
- This notebook will load the checkpoint when `use_checkpoint_mrbno=True`


## 1. Load Dataset and Imports


In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import sys
import os

# Add models directory to path
sys.path.append('models')

from FNO_model import FNO2d
from MRBNO_lift_v3 import MRBNOLiftV3
from MRBNO_simple import PCAReducerMultiChannel
from PCANet import PCANet

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:
# Load Kolmogorov flow dataset
# NOTE: This dataset requires ~3GB RAM. For login nodes, consider using SLURM to submit to compute nodes.
# For memory-efficient loading, we process in chunks and convert to float32 immediately.

data_path = '/pscratch/sd/d/dvod/2D_NS_Re5000.npy'
data = np.load(data_path, mmap_mode='r')  # Memory-mapped to avoid loading full file at once

print(f"Dataset shape: {data.shape}")
print(f"Interpretation: (num_trajectories={data.shape[0]}, num_timesteps={data.shape[1]}, height={data.shape[2]}, width={data.shape[3]})")
print(f"Data dtype: {data.dtype}")

# Create input-output pairs: u(t) -> u(t+1)
# Each trajectory has 501 timesteps, so we get 500 pairs per trajectory
num_trajectories = data.shape[0]
num_timesteps = data.shape[1] - 1  # 500 pairs per trajectory
total_pairs = num_trajectories * num_timesteps

print(f"\nCreating input-output pairs...")
print(f"  Trajectories: {num_trajectories}")
print(f"  Pairs per trajectory: {num_timesteps}")
print(f"  Total pairs: {total_pairs:,}")

# Process in chunks to be memory-efficient, convert to float32 immediately
chunk_size = 50  # Process 50 trajectories at a time
X_chunks = []
Y_chunks = []

for chunk_start in range(0, num_trajectories, chunk_size):
    chunk_end = min(chunk_start + chunk_size, num_trajectories)
    X_chunk_list = []
    Y_chunk_list = []
    
    for traj_idx in range(chunk_start, chunk_end):
        trajectory = np.array(data[traj_idx], dtype=np.float32)  # Convert to float32 early
        for t in range(num_timesteps):
            X_chunk_list.append(trajectory[t])      # u(t)
            Y_chunk_list.append(trajectory[t+1])    # u(t+1)
    
    # Convert chunk to array immediately (float32)
    X_chunk = np.array(X_chunk_list, dtype=np.float32)
    Y_chunk = np.array(Y_chunk_list, dtype=np.float32)
    X_chunks.append(X_chunk)
    Y_chunks.append(Y_chunk)
    
    if (chunk_start // chunk_size + 1) % 2 == 0:
        print(f"  Processed {chunk_end}/{num_trajectories} trajectories...")

# Concatenate all chunks
X = np.concatenate(X_chunks, axis=0)  # (total_pairs, 64, 64)
Y = np.concatenate(Y_chunks, axis=0)  # (total_pairs, 64, 64)
X_chunks, Y_chunks = None, None  # Free memory

print(f"\nTotal pairs created: {len(X):,}")
print(f"Input X shape: {X.shape}, dtype: {X.dtype}")
print(f"Output Y shape: {Y.shape}, dtype: {Y.dtype}")

# Add channel dimension for input (1 channel for velocity field)
X = X[:, np.newaxis, :, :]  # (num_pairs, 1, 64, 64)

print(f"\nFinal shapes:")
print(f"Input X: {X.shape} (batch, channels, height, width)")
print(f"Output Y: {Y.shape} (batch, height, width)")
print(f"Memory usage: X={X.nbytes/(1024**3):.2f} GB, Y={Y.nbytes/(1024**3):.2f} GB")

# Check data statistics
print(f"\nData statistics:")
print(f"X: min={X.min():.4f}, max={X.max():.4f}, mean={X.mean():.4f}, std={X.std():.4f}")
print(f"Y: min={Y.min():.4f}, max={Y.max():.4f}, mean={Y.mean():.4f}, std={Y.std():.4f}")

# Memory check
import psutil
import os
process = psutil.Process(os.getpid())
mem_used = process.memory_info().rss / (1024**3)
print(f"\nCurrent memory usage: {mem_used:.2f} GB")
print(f"⚠️  NOTE: Training requires ~8-16 GB GPU memory. Consider using SLURM for compute nodes.")


Dataset shape: (200, 501, 64, 64)
Interpretation: (num_trajectories=200, num_timesteps=501, height=64, width=64)
Data dtype: float64

Creating input-output pairs...
  Trajectories: 200
  Pairs per trajectory: 500
  Total pairs: 100,000
  Processed 100/200 trajectories...
  Processed 200/200 trajectories...

Total pairs created: 100,000
Input X shape: (100000, 64, 64), dtype: float32
Output Y shape: (100000, 64, 64), dtype: float32

Final shapes:
Input X: (100000, 1, 64, 64) (batch, channels, height, width)
Output Y: (100000, 64, 64) (batch, height, width)
Memory usage: X=1.53 GB, Y=1.53 GB

Data statistics:
X: min=-13.2881, max=13.3311, mean=0.0000, std=2.2089
Y: min=-13.2881, max=13.3311, mean=0.0000, std=2.2092

Current memory usage: 7.72 GB
⚠️  NOTE: Training requires ~8-16 GB GPU memory. Consider using SLURM for compute nodes.


## 2. Normalize and Split Data


In [3]:
# Train/val/test split (70/20/10 like the original notebook)
actual_samples = len(X)
train_frac = 0.7
val_frac = 0.2
n_train = int(train_frac * actual_samples)
n_val = int(val_frac * actual_samples)

# Split indices first (memory efficient)
train_idx = slice(0, n_train)
val_idx = slice(n_train, n_train + n_val)
test_idx = slice(n_train + n_val, None)

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val = X[val_idx], Y[val_idx]
X_test, Y_test = X[test_idx], Y[test_idx]

print(f"\nTraining samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Test samples: {len(X_test):,}")

# Normalize using train statistics only (to avoid data leakage)
print("\nComputing normalization statistics from training data...")
x_mean = float(X_train.mean())
x_std = float(X_train.std())
y_mean = float(Y_train.mean())
y_std = float(Y_train.std())

print(f"Normalizing data...")
X_train = (X_train - x_mean) / (x_std + 1e-8)
Y_train = (Y_train - y_mean) / (y_std + 1e-8)
X_val = (X_val - x_mean) / (x_std + 1e-8)
Y_val = (Y_val - y_mean) / (y_std + 1e-8)
X_test = (X_test - x_mean) / (x_std + 1e-8)
Y_test = (Y_test - y_mean) / (y_std + 1e-8)

print(f"\nX normalization: mean={x_mean:.4f}, std={x_std:.4f}")
print(f"Y normalization: mean={y_mean:.4f}, std={y_std:.4f}")

# Memory check after normalization
import psutil
import os
process = psutil.Process(os.getpid())
mem_used = process.memory_info().rss / (1024**3)
print(f"\nMemory after normalization: {mem_used:.2f} GB")



Training samples: 70,000
Validation samples: 20,000
Test samples: 10,000

Computing normalization statistics from training data...
Normalizing data...

X normalization: mean=0.0000, std=2.2085
Y normalization: mean=0.0000, std=2.2087

Memory after normalization: 10.78 GB


## 3. Prepare Data and PCA


In [ ]:
# Convert to PyTorch tensors
X_train_t = torch.from_numpy(X_train).float()  # (n_train, 1, 128, 128)
Y_train_t = torch.from_numpy(Y_train).float().unsqueeze(1)  # (n_train, 1, 128, 128)
X_val_t = torch.from_numpy(X_val).float()  # (n_val, 1, 128, 128)
Y_val_t = torch.from_numpy(Y_val).float().unsqueeze(1)  # (n_val, 1, 128, 128)

print(f"Training data: X={X_train_t.shape}, Y={Y_train_t.shape}")
print(f"Validation data: X={X_val_t.shape}, Y={Y_val_t.shape}")

# Create data loaders
# Use smaller batch size for MRBNO to avoid OOM (can be adjusted per model)
batch_size_fno = 8  # Reduced for 128x128 resolution
batch_size_mrbno = 8  # Reduced for 128x128 resolution
batch_size_pcanet = 8  # Reduced for 128x128 resolution

train_loader_fno = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=batch_size_fno, shuffle=True)
val_loader_fno = DataLoader(TensorDataset(X_val_t, Y_val_t), batch_size=batch_size_fno, shuffle=False)

train_loader_mrbno = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=batch_size_mrbno, shuffle=True)
val_loader_mrbno = DataLoader(TensorDataset(X_val_t, Y_val_t), batch_size=batch_size_mrbno, shuffle=False)

train_loader_pcanet = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=batch_size_pcanet, shuffle=True)
val_loader_pcanet = DataLoader(TensorDataset(X_val_t, Y_val_t), batch_size=batch_size_pcanet, shuffle=False)

print(f"Batch sizes: FNO={batch_size_fno}, MRBNO={batch_size_mrbno}, PCA Net={batch_size_pcanet}")

# Compute PCA ONCE - shared by both MRBNO Lift v3 and PCA Net
# Note: For Kolmogorov flow, input is single-channel (velocity field)
# PCAReducerMultiChannel expects multi-channel input, so we pass it as (channels=1)
rb = PCAReducerMultiChannel(
    bc_weight=1.0, 
    boundary_threshold=0.1, 
    use_boundary_weighting=False,
).fit(X_train, Y_train)

epsilon = 1e-3
k_in, k_out = rb.k_from_cutoff(j_in=1-1*epsilon, j_out=1-epsilon)

print(f"\nPCA components (shared by MRBNO and PCA Net): k_in={k_in}, k_out={k_out}")
print(f"Boundary condition awareness: bc_weight={rb.bc_weight}, threshold={rb.boundary_threshold}")
print(f"PCA whiten: {getattr(rb, 'whiten', False)}")


Training data: X=torch.Size([70000, 1, 64, 64]), Y=torch.Size([70000, 1, 64, 64])
Validation data: X=torch.Size([20000, 1, 64, 64]), Y=torch.Size([20000, 1, 64, 64])
Batch sizes: FNO=32, MRBNO=32, PCA Net=32

PCA components (shared by MRBNO and PCA Net): k_in=1179, k_out=1071
Boundary condition awareness: bc_weight=1.0, threshold=0.1
PCA whiten: False


## 4. Training Function


In [5]:
def train_model(model, train_loader, val_loader, epochs=300, lr=2e-3, weight_decay=1e-4, max_grad_norm=3.0, 
                device='cpu', early_stopping_patience=50, early_stopping_min_delta=0.0):
    """Train model with cosine annealing LR schedule and early stopping."""
    import time
    start_time = time.time()
    
    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
        eps=1e-8
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0.0)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    
    # Check if validation set is empty
    has_val = len(val_loader.dataset) > 0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        
        # Validation phase
        if has_val:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    pred = model(xb)
                    loss = criterion(pred, yb)
                    val_loss += loss.item() * xb.size(0)
            val_loss /= len(val_loader.dataset)
        else:
            val_loss = float('inf')  # No validation data
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        # Early stopping logic
        improved = False
        if has_val:
            if val_loss < best_val_loss - early_stopping_min_delta:
                best_val_loss = val_loss
                best_epoch = epoch
                patience_counter = 0
                improved = True
            else:
                patience_counter += 1
        else:
            # If no validation, use training loss
            if train_loss < best_val_loss - early_stopping_min_delta:
                best_val_loss = train_loss
                best_epoch = epoch
                patience_counter = 0
                improved = True
            else:
                patience_counter += 1
        
        scheduler.step()
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            if has_val:
                print(f"[{epoch+1:03d}/{epochs}] train={train_loss:.4e} val={val_loss:.4e} lr={scheduler.get_last_lr()[0]:.2e} (best: {best_val_loss:.4e} @ epoch {best_epoch+1})")
            else:
                print(f"[{epoch+1:03d}/{epochs}] train={train_loss:.4e} (no val) lr={scheduler.get_last_lr()[0]:.2e} (best: {best_val_loss:.4e} @ epoch {best_epoch+1})")
        
        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch+1} (patience={early_stopping_patience})")
            break
    
    training_time = time.time() - start_time
    
    return train_losses, val_losses, best_val_loss, training_time, best_epoch + 1


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else 
                      "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Checkpoint configuration - set to True to load existing checkpoints (skip training)
# Set to False to train from scratch
# NOTE: MRBNO should be trained using the separate script: train_mrbno_kolmogorov_Re5000.py
#       Then set use_checkpoint_mrbno=True to load the checkpoint in this notebook
use_checkpoint_fno = True
use_checkpoint_mrbno = True
use_checkpoint_pcanet = True

checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)


Using device: cuda


## 5a. Train FNO Model


In [ ]:
# Train or load FNO model
# Note: For Kolmogorov flow, input is 1 channel (velocity field)
print("\n" + "="*70)
print("FNO MODEL")
print("="*70)
fno_model = FNO2d(in_channels=1, out_channels=1, width=24, modes=(12, 12), n_layers=4)
print(f"FNO Parameters: {sum(p.numel() for p in fno_model.parameters()):,}")

fno_checkpoint_path = os.path.join(checkpoint_dir, 'fno_kolmogorov_Re5000_best.pt')

if use_checkpoint_fno and os.path.exists(fno_checkpoint_path):
    print(f"Loading FNO checkpoint from {fno_checkpoint_path}...")
    fno_model.load_state_dict(torch.load(fno_checkpoint_path, map_location=device))
    fno_model.to(device)
    fno_model.eval()
    print("FNO checkpoint loaded. Skipping training.")
    # Evaluate to get validation loss
    criterion = nn.MSELoss()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader_fno:
            xb, yb = xb.to(device), yb.to(device)
            pred = fno_model(xb)
            loss = criterion(pred, yb)
            val_loss += loss.item() * xb.size(0)
    fno_best_val = val_loss / len(val_loader_fno.dataset)
    fno_train_losses, fno_val_losses = [], []  # Not available from checkpoint
    fno_training_time = 0.0
    fno_best_epoch = 0
    print(f"FNO validation loss: {fno_best_val:.4e}")
else:
    print("Training FNO from scratch...")
    fno_train_losses, fno_val_losses, fno_best_val, fno_training_time, fno_best_epoch = train_model(
        fno_model, train_loader_fno, val_loader_fno, epochs=300, lr=2e-3, max_grad_norm=3.0, 
        device=device, early_stopping_patience=50
    )
    torch.save(fno_model.state_dict(), fno_checkpoint_path)
    print(f"FNO model saved to {fno_checkpoint_path}")
    print(f"Training time: {fno_training_time:.2f} seconds ({fno_training_time/60:.2f} minutes)")



FNO MODEL
FNO Parameters: 335,041
Training FNO from scratch...
[001/300] train=6.6313e-02 val=3.2996e-02 lr=2.00e-03 (best: 3.2996e-02 @ epoch 1)
[010/300] train=9.8463e-03 val=1.2714e-02 lr=1.99e-03 (best: 1.2714e-02 @ epoch 10)
[020/300] train=6.8687e-03 val=9.2256e-03 lr=1.98e-03 (best: 9.2256e-03 @ epoch 20)
[030/300] train=5.6680e-03 val=8.4560e-03 lr=1.95e-03 (best: 8.4560e-03 @ epoch 30)
[040/300] train=4.9993e-03 val=7.8396e-03 lr=1.91e-03 (best: 7.7463e-03 @ epoch 38)
[050/300] train=4.5624e-03 val=7.3382e-03 lr=1.87e-03 (best: 7.3382e-03 @ epoch 50)
[060/300] train=4.2318e-03 val=7.1208e-03 lr=1.81e-03 (best: 6.9428e-03 @ epoch 56)
[070/300] train=3.9424e-03 val=6.8558e-03 lr=1.74e-03 (best: 6.6648e-03 @ epoch 69)
[080/300] train=3.7517e-03 val=6.5573e-03 lr=1.67e-03 (best: 6.5573e-03 @ epoch 80)
[090/300] train=3.5405e-03 val=6.4777e-03 lr=1.59e-03 (best: 6.3333e-03 @ epoch 88)
[100/300] train=3.3751e-03 val=6.3265e-03 lr=1.50e-03 (best: 6.2786e-03 @ epoch 98)
[110/300] tra

## 5b. Train MRBNO Lift v3 Model


In [ ]:
# Train or load MRBNO Lift v3 model (using shared PCA reducer)
print("\n" + "="*70)
print("MRBNO LIFT V3 MODEL")
print("="*70)

# Clear GPU memory before training MRBNO (large model)
if torch.cuda.is_available():
    print("Clearing GPU memory before MRBNO training...")
    # Move FNO model to CPU to free GPU memory
    if 'fno_model' in locals() and fno_model is not None:
        fno_model.cpu()
        del fno_model
    # Clear PyTorch cache
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
    # Show memory status
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3  # GB
        reserved = torch.cuda.memory_reserved(0) / 1024**3   # GB
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3  # GB
        print(f"GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved, {total:.2f} GB total")
        print(f"GPU Memory free: {total - reserved:.2f} GB")
    print("GPU memory cleared.\n")

mrbno_lift_v3_model = MRBNOLiftV3(
    width=24,
    rb=rb,
    k_in=k_in,
    k_out=k_out,
    layers=4,
    dropout=0.0,
    init_scale=0.1,
    use_skip_connections=True,
    use_batch_norm=True,
    use_coordinates=True,
)

mrbno_lift_v3_params = sum(p.numel() for p in mrbno_lift_v3_model.parameters())
print(f"MRBNO Lift v3 Parameters: {mrbno_lift_v3_params:,}")

mrbno_checkpoint_path = os.path.join(checkpoint_dir, 'mrbno_lift_v3_kolmogorov_Re5000_best.pt')

if use_checkpoint_mrbno and os.path.exists(mrbno_checkpoint_path):
    print(f"Loading MRBNO checkpoint from {mrbno_checkpoint_path}...")
    mrbno_lift_v3_model.load_state_dict(torch.load(mrbno_checkpoint_path, map_location=device))
    mrbno_lift_v3_model.to(device)
    mrbno_lift_v3_model.eval()
    print("MRBNO checkpoint loaded. Skipping training.")
    # Evaluate to get validation loss
    criterion = nn.MSELoss()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader_mrbno:
            xb, yb = xb.to(device), yb.to(device)
            pred = mrbno_lift_v3_model(xb)
            loss = criterion(pred, yb)
            val_loss += loss.item() * xb.size(0)
    mrbno_lift_v3_best_val = val_loss / len(val_loader_mrbno.dataset)
    mrbno_lift_v3_train_losses, mrbno_lift_v3_val_losses = [], []  # Not available from checkpoint
    mrbno_lift_v3_training_time = 0.0
    mrbno_lift_v3_best_epoch = 0
    print(f"MRBNO validation loss: {mrbno_lift_v3_best_val:.4e}")
else:
    print("Training MRBNO from scratch...")
    print(f"Using reduced batch size {batch_size_mrbno} to avoid OOM")
    
    # Additional memory clearing right before moving model to GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"Final GPU memory check before model.to(device)...")
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  Available: {total - reserved:.2f} GB")
    mrbno_lift_v3_train_losses, mrbno_lift_v3_val_losses, mrbno_lift_v3_best_val, mrbno_lift_v3_training_time, mrbno_lift_v3_best_epoch = train_model(
        mrbno_lift_v3_model, train_loader_mrbno, val_loader_mrbno, epochs=300, lr=2e-3, max_grad_norm=3.0, weight_decay=1e-4, 
        device=device, early_stopping_patience=50
    )
    torch.save(mrbno_lift_v3_model.state_dict(), mrbno_checkpoint_path)
    print(f"MRBNO Lift v3 model saved to {mrbno_checkpoint_path}")
    print(f"Training time: {mrbno_lift_v3_training_time:.2f} seconds ({mrbno_lift_v3_training_time/60:.2f} minutes)")



MRBNO LIFT V3 MODEL
Clearing GPU memory before MRBNO training...
GPU Memory: 0.02 GB allocated, 0.02 GB reserved, 39.38 GB total
GPU Memory free: 39.36 GB
GPU memory cleared.

MRBNO Lift v3 Parameters: 3,510,085,458
Training MRBNO from scratch...
Using reduced batch size 32 to avoid OOM
Final GPU memory check before model.to(device)...
  Available: 39.36 GB


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.46 GiB. GPU 0 has a total capacity of 39.38 GiB of which 1.10 GiB is free. Including non-PyTorch memory, this process has 38.27 GiB memory in use. Of the allocated memory 37.72 GiB is allocated by PyTorch, and 41.64 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5c. Train PCA Net Model


In [ ]:
# Train or load PCA Net model (using shared PCA reducer)
print("\n" + "="*70)
print("PCA NET MODEL")
print("="*70)

pcanet_model = PCANet(
    rb=rb,
    k_in=k_in,
    k_out=k_out,
    hidden=1024,
    depth=5,
    dropout=0.0,
    init_scale=0.1,
)

pcanet_params = sum(p.numel() for p in pcanet_model.parameters())
print(f"PCA Net Parameters: {pcanet_params:,}")

pcanet_checkpoint_path = os.path.join(checkpoint_dir, 'pcanet_kolmogorov_Re5000_best.pt')

if use_checkpoint_pcanet and os.path.exists(pcanet_checkpoint_path):
    print(f"Loading PCA Net checkpoint from {pcanet_checkpoint_path}...")
    pcanet_model.load_state_dict(torch.load(pcanet_checkpoint_path, map_location=device))
    pcanet_model.to(device)
    pcanet_model.eval()
    print("PCA Net checkpoint loaded. Skipping training.")
    # Evaluate to get validation loss
    criterion = nn.MSELoss()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader_pcanet:
            xb, yb = xb.to(device), yb.to(device)
            pred = pcanet_model(xb)
            loss = criterion(pred, yb)
            val_loss += loss.item() * xb.size(0)
    pcanet_best_val = val_loss / len(val_loader_pcanet.dataset)
    pcanet_train_losses, pcanet_val_losses = [], []  # Not available from checkpoint
    pcanet_training_time = 0.0
    pcanet_best_epoch = 0
    print(f"PCA Net validation loss: {pcanet_best_val:.4e}")
else:
    print("Training PCA Net from scratch...")
    pcanet_train_losses, pcanet_val_losses, pcanet_best_val, pcanet_training_time, pcanet_best_epoch = train_model(
        pcanet_model, train_loader_pcanet, val_loader_pcanet, epochs=300, lr=2e-3, max_grad_norm=3.0, weight_decay=1e-4, 
        device=device, early_stopping_patience=50
    )
    torch.save(pcanet_model.state_dict(), pcanet_checkpoint_path)
    print(f"PCA Net model saved to {pcanet_checkpoint_path}")
    print(f"Training time: {pcanet_training_time:.2f} seconds ({pcanet_training_time/60:.2f} minutes)")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"\nFinal Results Summary:")
print(f"  FNO:           best val loss = {fno_best_val:.4e} (epoch {fno_best_epoch})")
print(f"  MRBNO Lift v3: best val loss = {mrbno_lift_v3_best_val:.4e} (epoch {mrbno_lift_v3_best_epoch})")
print(f"  PCA Net:       best val loss = {pcanet_best_val:.4e} (epoch {pcanet_best_epoch})")


## 6. Training Results and Comparison


In [ ]:
# Compare results from all models and compute relative L1 errors with statistics
print("\n" + "="*70)
print("TRAINING COMPLETE - SUMMARY")
print("="*70)

# Enhanced error computation function with per-sample errors
def compute_rel_l1_error_stats(model, val_loader, device, y_mean, y_std):
    """
    Compute relative L1 error statistics:
    - Mean relative L1 error: ||u_pred - u_true||_1 / ||u_true||_1
    - Standard deviation across samples
    - Average spatial gradient magnitude of relative error (measures spatial smoothness)
    - Per-sample errors (returned for analysis)
    """
    model.eval()
    per_sample_errors = []
    spatial_grad_magnitudes = []
    
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            
            # Denormalize
            pred_denorm = pred * y_std + y_mean
            yb_denorm = yb * y_std + y_mean
            
            batch_size = pred_denorm.size(0)
            
            # Compute per-sample relative L1 errors
            for i in range(batch_size):
                pred_sample = pred_denorm[i].cpu().numpy()  # (1, H, W) or (H, W)
                true_sample = yb_denorm[i].cpu().numpy()
                
                # Handle different shapes
                if pred_sample.ndim == 3:
                    pred_sample = pred_sample[0]
                    true_sample = true_sample[0]
                
                # Relative L1 error per sample
                l1_error = np.abs(pred_sample - true_sample).sum()
                l1_norm_true = np.abs(true_sample).sum()
                rel_l1 = l1_error / (l1_norm_true + 1e-10)
                per_sample_errors.append(rel_l1)
                
                # Spatial gradient magnitude of relative error (measures spatial smoothness)
                # Higher values = more spatial variation (jagged/noisy errors)
                # Lower values = smoother errors (spatially correlated)
                error_map = np.abs(pred_sample - true_sample)
                true_abs = np.abs(true_sample)
                rel_error_map = error_map / (true_abs + 1e-10)  # Relative error at each pixel
                
                # Compute spatial gradient
                grad_x, grad_y = np.gradient(rel_error_map)
                grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)
                mean_grad_magnitude = np.mean(grad_magnitude)
                spatial_grad_magnitudes.append(mean_grad_magnitude)
    
    per_sample_errors = np.array(per_sample_errors)
    spatial_grad_magnitudes = np.array(spatial_grad_magnitudes)
    
    return {
        'mean': np.mean(per_sample_errors),
        'std': np.std(per_sample_errors),
        'spatial_grad_mean': np.mean(spatial_grad_magnitudes),
        'spatial_grad_std': np.std(spatial_grad_magnitudes),
        'per_sample_errors': per_sample_errors,
        'spatial_grad_magnitudes': spatial_grad_magnitudes
    }

# Get parameter counts
fno_params = sum(p.numel() for p in fno_model.parameters())
mrbno_lift_v3_params = sum(p.numel() for p in mrbno_lift_v3_model.parameters())
pcanet_params = sum(p.numel() for p in pcanet_model.parameters())

# Compute enhanced error statistics (use appropriate loaders)
fno_stats = compute_rel_l1_error_stats(fno_model, val_loader_fno, device, y_mean, y_std)
mrbno_lift_v3_stats = compute_rel_l1_error_stats(mrbno_lift_v3_model, val_loader_mrbno, device, y_mean, y_std)
pcanet_stats = compute_rel_l1_error_stats(pcanet_model, val_loader_pcanet, device, y_mean, y_std)

# Measure inference time
import time
def measure_inference_time(model, val_loader, device, n_runs=10):
    """Measure average inference time per batch"""
    model.eval()
    times = []
    
    with torch.no_grad():
        for _ in range(n_runs):
            for xb, yb in val_loader:
                xb = xb.to(device)
                torch.cuda.synchronize() if device.type == 'cuda' else None
                start = time.time()
                _ = model(xb)
                torch.cuda.synchronize() if device.type == 'cuda' else None
                times.append(time.time() - start)
                break  # Only measure first batch
    return np.mean(times)

fno_inference_time = measure_inference_time(fno_model, val_loader_fno, device)
mrbno_lift_v3_inference_time = measure_inference_time(mrbno_lift_v3_model, val_loader_mrbno, device)
pcanet_inference_time = measure_inference_time(pcanet_model, val_loader_pcanet, device)

models_summary = [
    ("FNO", fno_train_losses, fno_val_losses, fno_best_val, fno_params, 
     fno_training_time, fno_inference_time, fno_best_epoch),
    ("MRBNO Lift v3", mrbno_lift_v3_train_losses, mrbno_lift_v3_val_losses, mrbno_lift_v3_best_val, 
     mrbno_lift_v3_params, mrbno_lift_v3_training_time, mrbno_lift_v3_inference_time, mrbno_lift_v3_best_epoch),
    ("PCA Net", pcanet_train_losses, pcanet_val_losses, pcanet_best_val, pcanet_params,
     pcanet_training_time, pcanet_inference_time, pcanet_best_epoch),
]

stats_list = [fno_stats, mrbno_lift_v3_stats, pcanet_stats]

print(f"\n{'Model':<20} {'Params':<12} {'Best Val Loss':<15} {'Rel L1 Error (mean±std)':<25} {'Spatial Grad Mag (mean±std)':<30} {'Train Time (min)':<18} {'Inf Time (ms)':<15} {'Best Epoch':<12}")
print("-" * 155)

for i, (model_name, train_losses, val_losses, best_val, params, train_time, inf_time, best_epoch) in enumerate(models_summary):
    stats = stats_list[i]
    rel_l1_str = f"{stats['mean']*100:.2f}% ± {stats['std']*100:.2f}%"
    spatial_grad_str = f"{stats['spatial_grad_mean']:.4e} ± {stats['spatial_grad_std']:.4e}"
    print(f"{model_name:<20} {params:>10,}  {best_val:>12.4e}  {rel_l1_str:<25}  {spatial_grad_str:<30}  {train_time/60:>16.2f}  {inf_time*1000:>13.2f}  {best_epoch:>10}")

print("\n" + "="*70)
print("DETAILED ERROR STATISTICS")
print("="*70)
print(f"\nFNO:")
print(f"  Relative L1 Error: {fno_stats['mean']*100:.4f}% ± {fno_stats['std']*100:.4f}%")
print(f"  Mean Spatial Gradient Magnitude: {fno_stats['spatial_grad_mean']:.6e} ± {fno_stats['spatial_grad_std']:.6e}")
print(f"    (Higher = more spatial variation/jagged errors, Lower = smoother errors)")
print(f"\nMRBNO Lift v3:")
print(f"  Relative L1 Error: {mrbno_lift_v3_stats['mean']*100:.4f}% ± {mrbno_lift_v3_stats['std']*100:.4f}%")
print(f"  Mean Spatial Gradient Magnitude: {mrbno_lift_v3_stats['spatial_grad_mean']:.6e} ± {mrbno_lift_v3_stats['spatial_grad_std']:.6e}")
print(f"    (Higher = more spatial variation/jagged errors, Lower = smoother errors)")
print(f"\nPCA Net:")
print(f"  Relative L1 Error: {pcanet_stats['mean']*100:.4f}% ± {pcanet_stats['std']*100:.4f}%")
print(f"  Mean Spatial Gradient Magnitude: {pcanet_stats['spatial_grad_mean']:.6e} ± {pcanet_stats['spatial_grad_std']:.6e}")
print(f"    (Higher = more spatial variation/jagged errors, Lower = smoother errors)")


In [ ]:
# Compare per-sample errors between FNO and MRBNO
fno_errors = fno_stats['per_sample_errors']
mrbno_errors = mrbno_lift_v3_stats['per_sample_errors']

# Find samples where each model performs better
fno_better = fno_errors < mrbno_errors
mrbno_better = mrbno_errors < fno_errors
tie = np.abs(fno_errors - mrbno_errors) < 1e-6  # Very close errors

print("="*70)
print("MODEL PERFORMANCE COMPARISON (FNO vs MRBNO Lift v3)")
print("="*70)
print(f"\nFNO performs better: {np.sum(fno_better)} samples ({np.sum(fno_better)/len(fno_better)*100:.1f}%)")
print(f"MRBNO performs better: {np.sum(mrbno_better)} samples ({np.sum(mrbno_better)/len(mrbno_better)*100:.1f}%)")
print(f"Similar performance: {np.sum(tie)} samples")

# Compute properties for all validation samples
def compute_sample_properties(X_val_t, Y_val_t, device):
    """Compute various properties of samples that might explain model performance differences."""
    properties = {
        'input_variance': [],
        'input_magnitude': [],
        'output_variance': [],
        'output_magnitude': [],
        'output_gradient_magnitude': [],  # Spatial gradient ||∇u||
        'output_laplacian_magnitude': [],  # ||Δu||
        'high_freq_content': [],  # Energy in high frequencies
        'temporal_change_magnitude': [],  # For time-series: |u(t+1) - u(t)| / |u(t)|
    }
    
    # Convert to numpy for processing
    X_val_np = X_val_t.cpu().numpy()
    Y_val_np = Y_val_t.cpu().numpy()
    
    for i in range(len(X_val_np)):
        x_sample = X_val_np[i]  # (1, H, W) for single-channel velocity field
        y_sample = Y_val_np[i]  # (1, H, W) or (H, W)
        
        # Handle different shapes
        if x_sample.ndim == 3:
            x_sample = x_sample[0]  # Take first (and only) channel
        if y_sample.ndim == 3:
            y_sample = y_sample[0]
        elif y_sample.ndim == 2:
            pass  # Already 2D
        
        # Input properties
        properties['input_variance'].append(np.var(x_sample))
        properties['input_magnitude'].append(np.abs(x_sample).mean())
        
        # Output properties
        properties['output_variance'].append(np.var(y_sample))
        properties['output_magnitude'].append(np.abs(y_sample).mean())
        
        # Spatial gradient (finite difference)
        grad_y = np.gradient(y_sample)
        grad_magnitude = np.sqrt(grad_y[0]**2 + grad_y[1]**2)
        properties['output_gradient_magnitude'].append(np.mean(grad_magnitude))
        
        # Laplacian (second derivatives)
        laplacian = np.gradient(np.gradient(y_sample)[0])[0] + np.gradient(np.gradient(y_sample)[1])[1]
        properties['output_laplacian_magnitude'].append(np.abs(laplacian).mean())
        
        # Temporal change magnitude (for Kolmogorov flow: |u(t+1) - u(t)| / |u(t)|)
        # Since input is u(t) and output is u(t+1)
        temporal_change = np.abs(y_sample - x_sample) / (np.abs(x_sample) + 1e-10)
        properties['temporal_change_magnitude'].append(np.mean(temporal_change))
        
        # High-frequency content (via FFT)
        fft_y = np.fft.fft2(y_sample)
        fft_magnitude = np.abs(fft_y)
        # High frequencies are outer region of FFT
        H, W = y_sample.shape
        center_h, center_w = H // 2, W // 2
        mask = np.ones((H, W), dtype=bool)
        # Low frequencies are near center
        mask[center_h-H//4:center_h+H//4, center_w-W//4:center_w+W//4] = False
        high_freq_energy = fft_magnitude[mask].sum()
        total_energy = fft_magnitude.sum()
        properties['high_freq_content'].append(high_freq_energy / (total_energy + 1e-10))
    
    # Convert to numpy arrays
    for key in properties:
        properties[key] = np.array(properties[key])
    
    return properties

print("\nComputing sample properties...")
sample_properties = compute_sample_properties(X_val_t, Y_val_t, device)

# Compare properties between samples where FNO wins vs MRBNO wins
print("\n" + "="*70)
print("PROPERTY ANALYSIS: FNO Better vs MRBNO Better Samples")
print("="*70)

property_names = [
    'input_variance', 'input_magnitude', 'output_variance', 'output_magnitude',
    'output_gradient_magnitude', 'output_laplacian_magnitude',
    'high_freq_content', 'temporal_change_magnitude'
]

print(f"\n{'Property':<30} {'FNO Better (mean±std)':<30} {'MRBNO Better (mean±std)':<30} {'Difference':<20}")
print("-" * 110)

for prop_name in property_names:
    fno_better_vals = sample_properties[prop_name][fno_better]
    mrbno_better_vals = sample_properties[prop_name][mrbno_better]
    
    fno_mean = np.mean(fno_better_vals) if len(fno_better_vals) > 0 else 0
    fno_std = np.std(fno_better_vals) if len(fno_better_vals) > 0 else 0
    mrbno_mean = np.mean(mrbno_better_vals) if len(mrbno_better_vals) > 0 else 0
    mrbno_std = np.std(mrbno_better_vals) if len(mrbno_better_vals) > 0 else 0
    
    diff = mrbno_mean - fno_mean
    diff_pct = (diff / (fno_mean + 1e-10)) * 100 if fno_mean > 0 else 0
    
    fno_str = f"{fno_mean:.4e} ± {fno_std:.4e}" if len(fno_better_vals) > 0 else "N/A"
    mrbno_str = f"{mrbno_mean:.4e} ± {mrbno_std:.4e}" if len(mrbno_better_vals) > 0 else "N/A"
    diff_str = f"{diff:.4e} ({diff_pct:+.1f}%)"
    
    print(f"{prop_name:<30} {fno_str:<30} {mrbno_str:<30} {diff_str:<20}")

# Identify key distinguishing factors
print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

# Find properties with largest relative differences
relative_diffs = {}
for prop_name in property_names:
    fno_vals = sample_properties[prop_name][fno_better]
    mrbno_vals = sample_properties[prop_name][mrbno_better]
    if len(fno_vals) > 0 and len(mrbno_vals) > 0:
        fno_mean = np.mean(fno_vals)
        mrbno_mean = np.mean(mrbno_vals)
        if fno_mean > 0:
            rel_diff = abs((mrbno_mean - fno_mean) / fno_mean)
            relative_diffs[prop_name] = rel_diff

# Sort by relative difference
sorted_props = sorted(relative_diffs.items(), key=lambda x: x[1], reverse=True)

print("\nProperties that most distinguish FNO-better vs MRBNO-better samples:")
for i, (prop_name, rel_diff) in enumerate(sorted_props[:5], 1):
    fno_vals = sample_properties[prop_name][fno_better]
    mrbno_vals = sample_properties[prop_name][mrbno_better]
    fno_mean = np.mean(fno_vals)
    mrbno_mean = np.mean(mrbno_vals)
    print(f"{i}. {prop_name}:")
    print(f"   FNO-better samples: {fno_mean:.4e}")
    print(f"   MRBNO-better samples: {mrbno_mean:.4e}")
    print(f"   Relative difference: {rel_diff*100:.1f}%")
    
    # Interpretation
    if mrbno_mean > fno_mean:
        print(f"   → MRBNO performs better on samples with HIGHER {prop_name}")
    else:
        print(f"   → MRBNO performs better on samples with LOWER {prop_name}")

# Visualization: Scatter plot of error difference vs properties
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

error_diff = mrbno_errors - fno_errors  # Positive means MRBNO worse, negative means MRBNO better

for i, prop_name in enumerate(property_names[:8]):
    ax = axes[i]
    prop_vals = sample_properties[prop_name]
    
    # Color by which model performs better
    colors = ['green' if e < 0 else 'blue' for e in error_diff]
    ax.scatter(prop_vals, error_diff, c=colors, alpha=0.5, s=10)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel(prop_name)
    ax.set_ylabel('Error Diff (MRBNO - FNO)')
    ax.set_title(f'{prop_name} vs Error Difference')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
